# 02 - Results
Loads the JSON reports produced by the pipeline and renders tables/plots. Run after `make train abtest`.

In [ ]:
import sys, json
sys.path.insert(0,'../src')
import pandas as pd
from pathlib import Path
from recsys.config import load_config, path
cfg = load_config(); R = Path(path(cfg,'reports'))
load = lambda n: json.loads((R/n).read_text()) if (R/n).exists() else None

## Retrieval

In [ ]:
r = load('retrieval_metrics.json')
pd.DataFrame({k: r[k] for k in ('popularity','svd_mf','two_tower')}).T[['recall@10','recall@50','recall@100','ndcg@10']] if r else 'not run'

## Ranking vs retrieval-only

In [ ]:
k = load('ranking_metrics.json')
if k:
    rows = {'retrieval_only': next(iter(k['models'].values()))['retrieval_only'], **{n: v['ranker'] for n, v in k['models'].items()}}
    display(pd.DataFrame(rows).T[['ndcg@10','recall@10','map@10']])
    display(pd.DataFrame(k['ablations']).T)

## A/B (simulated)

In [ ]:
a = load('ab_results.json')
a and (pd.DataFrame({n: {'CTR': m['ctr'], 'NDCG@10': m['ndcg@10']['mean'], 'users': a['n_users'][n]} for n, m in a['arms'].items()}), a['decision'])